## **Aim**
To implement a program that analyzes network traffic logs to detect port-scanning activity by identifying a single source IP attempting to connect to many different ports in a short time.

## **Algorithm**
**Step 1:** Create a dummy network log file (CSV style: `timestamp, source_ip, dest_ip, dest_port`).

**Step 2:** Use a dictionary to track the set of unique ports accessed by each source IP.

**Step 3:** Read the log file and populate the dictionary.

**Step 4:** Define a threshold for the number of unique ports (e.g., 10 unique ports) that indicates a scan.

**Step 5:** Iterate through the source IPs and identify those whose unique port count exceeds the threshold.

**Step 6:** Output the list of IPs flagged as "Port Scanners".

In [1]:
from collections import defaultdict

def detect_port_scanning(log_file, threshold=10):
    # Maps source_ip -> set of unique dest_ports
    scanner_map = defaultdict(set)
    
    with open(log_file, "r") as f:
        next(f) # Skip header
        for line in f:
            parts = line.strip().split(",")
            if len(parts) == 4:
                _, src_ip, _, port = parts
                scanner_map[src_ip].add(port)
    
    flagged_ips = {ip: len(ports) for ip, ports in scanner_map.items() if len(ports) >= threshold}
    return flagged_ips

def main():
    log_file = "network_traffic.log"
    
    # Create dummy network log
    with open(log_file, "w") as f:
        f.write("timestamp,src_ip,dst_ip,dst_port\n")
        # Normal traffic
        f.write("2026-08-06 10:00,192.168.1.5,192.168.1.1,80\n")
        f.write("2026-08-06 10:01,192.168.1.5,192.168.1.1,443\n")
        # Port scan from 10.0.0.20
        for port in range(21, 31): # Scanning ports 21 to 30
            f.write(f"2026-08-06 10:05,10.0.0.20,192.168.1.1,{port}\n")
            
    print(f"Analyzing {log_file} for port scanning...")
    scanners = detect_port_scanning(log_file)
    
    if scanners:
        print("\nPORT SCANNING DETECTED:")
        for ip, port_count in scanners.items():
            print(f"Source IP: {ip} | Unique Ports Targeted: {port_count}")
    else:
        print("\nNo port scanning activity detected.")

if __name__ == "__main__":
    main()

Analyzing network_traffic.log for port scanning...

PORT SCANNING DETECTED:
Source IP: 10.0.0.20 | Unique Ports Targeted: 10


## **Result**
This the program successfully analyzes a simulated network traffic log and detects port-scanning behavior.